# 广播语义
许多PyTorch操作支持NumPy的广播语义。
详情请见 https://numpy.org/doc/stable/user/basics.broadcasting.html。
简而言之，如果一个PyTorch操作支持广播，那么它的Tensor参数可以自动扩展为相同的大小（无需复制数据）。


## 通用广播规则
```
General broadcasting rules
When operating on two arrays, NumPy compares their shapes element-wise. It starts with the trailing (i.e. rightmost) dimension and works its way left. Two dimensions are compatible when

they are equal, or
one of them is 1.
```
在对两个数组进行操作时，NumPy会逐元素比较它们的形状。它从尾部（即最右侧）维度开始，向左进行比较。当两个维度满足以下条件时，它们是兼容的：

它们相等，或者
其中一个维度为1。
一般语义
两个张量是“可广播”的，如果满足以下规则：
- 每个张量至少有一个维度。（官方不严谨，标量是支持的）
- 在遍历维度大小时，从尾部维度开始， 维度大小必须相等，或者其中一个为1，或者其中一个不存在。（就是sharp）

## 广播的完整规则

1. 判断：从右向左比较两个张量的 shape，缺失的维度在左边补 1。每个维度要么大小相同，要么其中一个为 1（左右都可以）。一旦出现既不相同也不是 1 的情况，直接报错。
2. 结果形状：广播成功后，结果的形状是每个维度取两个数字中较大的那个（非 1 的那个）。
3. 扩展：左右两边都有可能广播，每个张量都把自己扩展成最终结果形状。
4. 数据填充：从右向左，连续匹配的那一块数据会被复制到缺失维度上。

---
📌 一句话终极版
“从右向左，缺则补 1，相同或 1 则过，否则报错；结果取大，两边都扩，数据只来源于最右边连续的那一块。且在匹配范围内一定有且只有一块”

# 示例

## AB双广播

In [9]:
import torch
A = torch.tensor([[[1, 2, 3, 4]],  # shape (3, 1, 4) 最大取 3,2,4
                  [[21, 22, 33, 44]],
                  [[31, 32, 33, 34]]])
print("A shape=",A.shape)
B = torch.tensor([[10, 20, 30, 40],    # shape (2, 4) =>(1,2,4) 取最大为 3,2,4
                  [100, 200, 300, 400]])
print("B shape=",B.shape)
C = A + B
print(C)
print("C shape=",C.shape)
A_expanded = [
    # 第 0 层（原始 A[0]）
    [[1, 2, 3, 4],      # 原始行
     [1, 2, 3, 4]],     # 复制出来的行（和上面一模一样）

    # 第 1 层（原始 A[1]）
    [[21, 22, 33, 44],  # 原始行
     [21, 22, 33, 44]], # 复制出来的行

    # 第 2 层（原始 A[2]）
    [[31, 32, 33, 34],  # 原始行
     [31, 32, 33, 34]]  # 复制出来的行
]
A_expanded = torch.tensor(A_expanded)
B_expanded = [
    # 第 0 层（原始 B 的第 0 行和第 1 行）
    [[10, 20, 30, 40],   # 原始 B[0]
     [100, 200, 300, 400]], # 原始 B[1]

    # 第 1 层（完全复制第 0 层）
    [[10, 20, 30, 40],
     [100, 200, 300, 400]],

    # 第 2 层（完全复制第 0 层）
    [[10, 20, 30, 40],
     [100, 200, 300, 400]]
]
B_expanded = torch.tensor(B_expanded)

print("A_expanded shape=",A_expanded.shape)
print("B_expanded shape=",B_expanded.shape)

C_EXPANDED = A_expanded + B_expanded
print("C_expanded shape=",C_EXPANDED)

print(C==C_EXPANDED)

A shape= torch.Size([3, 1, 4])
B shape= torch.Size([2, 4])
tensor([[[ 11,  22,  33,  44],
         [101, 202, 303, 404]],

        [[ 31,  42,  63,  84],
         [121, 222, 333, 444]],

        [[ 41,  52,  63,  74],
         [131, 232, 333, 434]]])
C shape= torch.Size([3, 2, 4])
A_expanded shape= torch.Size([3, 2, 4])
B_expanded shape= torch.Size([3, 2, 4])
C_expanded shape= tensor([[[ 11,  22,  33,  44],
         [101, 202, 303, 404]],

        [[ 31,  42,  63,  84],
         [121, 222, 333, 444]],

        [[ 41,  52,  63,  74],
         [131, 232, 333, 434]]])
tensor([[[True, True, True, True],
         [True, True, True, True]],

        [[True, True, True, True],
         [True, True, True, True]],

        [[True, True, True, True],
         [True, True, True, True]]])
